In [46]:
import pandas as pd
import numpy as np
import itertools
import joblib
import random
import scipy.stats as stats
import time

### Due to not having Sufficient Data on every player in the World Cup I decided on a custom World Cup and made 4 Pots similar to the Fifa draw though i did not put USA in Pot A due to my model not having any Home team Bias

In [47]:
prediction_engine = joblib.load('../../../models/Prediction_model/world_cup_stacked_engine.pkl')
chaos_std = joblib.load('../../../models/Prediction_model/residuals.pkl')

In [48]:
df=pd.read_csv('../../../data/processed/Teams/World_Cup_Nations_26.csv')

In [49]:
df.columns

Index(['team_name', 'international_strength', 'fifa_points_norm', 'elo_norm',
       'f_assists_per90_zscore', 'f_keypasses_per90_zscore',
       'f_bigchancescreated_per90_zscore', 'f_goalconversionpercentage_zscore',
       'f_defensive_actions_zscore', 'f_aerialduelswonpercentage_zscore',
       'm_shotsontarget_per90_zscore', 'm_possessionwonattthird_per90_zscore',
       'm_interceptions_per90_zscore', 'm_ballrecovery_per90_zscore',
       'm_defensive_actions_zscore', 'm_successfuldribbles_per90_zscore',
       'd_defensive_actions_zscore', 'd_interceptions_per90_zscore',
       'd_clearances_per90_zscore', 'd_tackleswon_per90_zscore',
       'd_aerialduelswonpercentage_zscore', 'd_accuratelongballs_per90_zscore',
       'd_possessionwonattthird_per90_zscore',
       'd_groundduelswonpercentage_zscore',
       'g_aerialduelswonpercentage_zscore', 'g_accuratelongballs_per90_zscore',
       'g_highclaims_per90_zscore', 'meta_n_forwards', 'meta_n_midfielders',
       'meta_n_defende

In [50]:
display(df[['team_name','team_id']])

,team_name,team_id
0,Ivory Coast,12
1,Switzerland,21
2,Belgium,4
3,France,10
4,Senegal,18
5,Brazil,5
6,United States,23
7,Saudi Arabia,17
8,Türkiye,22
9,Croatia,8


In [51]:
teams_data = [
    {"team_name": "Argentina", "team_id": 2, "fifa_rank": 1},
    {"team_name": "Spain", "team_id": 19, "fifa_rank": 2},
    {"team_name": "France", "team_id": 10, "fifa_rank": 3},
    {"team_name": "England", "team_id": 9, "fifa_rank": 4},
    {"team_name": "Portugal", "team_id": 16, "fifa_rank": 5},
    {"team_name": "Brazil", "team_id": 5, "fifa_rank": 6},
    {"team_name": "Netherlands", "team_id": 14, "fifa_rank": 8},
    {"team_name": "Belgium", "team_id": 4, "fifa_rank": 9},
    {"team_name": "Germany", "team_id": 11, "fifa_rank": 10},
    {"team_name": "Croatia", "team_id": 8, "fifa_rank": 11},
    {"team_name": "Colombia", "team_id": 7, "fifa_rank": 13},
    {"team_name": "Senegal", "team_id": 18, "fifa_rank": 15},
    {"team_name": "Uruguay", "team_id": 24, "fifa_rank": 16},
    {"team_name": "United States", "team_id": 23, "fifa_rank": 17},
    {"team_name": "Japan", "team_id": 13, "fifa_rank": 18},
    {"team_name": "Switzerland", "team_id": 21, "fifa_rank": 19},
    {"team_name": "Türkiye", "team_id": 22, "fifa_rank": 22},
    {"team_name": "Australia", "team_id": 3, "fifa_rank": 27},
    {"team_name": "Algeria", "team_id": 1, "fifa_rank": 28},
    {"team_name": "Canada", "team_id": 6, "fifa_rank": 30},
    {"team_name": "Norway", "team_id": 15, "fifa_rank": 31},
    {"team_name": "Ivory Coast", "team_id": 12, "fifa_rank": 33},
    {"team_name": "Sweden", "team_id": 20, "fifa_rank": 38},
    {"team_name": "Saudi Arabia", "team_id": 17, "fifa_rank": 61}
]
team_id_to_rank = {team["team_id"]: team["fifa_rank"] for team in teams_data}
teams_df = pd.DataFrame(teams_data)

In [52]:
df['fifa_rank']=df['team_id'].map(team_id_to_rank)

In [53]:
def build_match_features(team_A_row, team_B_row):

    metadata = ['team_name', 'team_id','fifa_rank']
    A_clean = team_A_row.drop(columns=metadata, errors='ignore')
    B_clean = team_B_row.drop(columns=metadata, errors='ignore')
    
    match_data = {}

    for col in A_clean.columns:
        val_A = A_clean[col].values[0]
        val_B = B_clean[col].values[0]
        

        match_data[f'home_{col}'] = val_A
   
        match_data[f'away_{col}'] = val_B

        match_data[f'diff_{col}'] = val_A - val_B
        
    return pd.DataFrame([match_data])

In [54]:
def build_prediction_matrix(df, prediction_engine):
    teams = df['team_name'].tolist()
    matrix = {team: {} for team in teams}
    
    expected_columns = prediction_engine.feature_names_in_
    
    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):
            tA, tB = teams[i], teams[j]
            
            row_A = df[df['team_name'] == tA]
            row_B = df[df['team_name'] == tB]
            
        
            features_A = build_match_features(row_A, row_B)
            features_A = features_A[expected_columns]
            
            gd_A = prediction_engine.predict(features_A)[0]
            matrix[tA][tB] = gd_A
            
          
            features_B = build_match_features(row_B, row_A)
            features_B = features_B[expected_columns]
            
            gd_B = prediction_engine.predict(features_B)[0]
            matrix[tB][tA] = gd_B
            
    return matrix

### Decided to make a Matrix of every Possible Matchup Predicted to save Computational Power

In [55]:
def draw_groups(df):
    df_sorted = df.sort_values(by='fifa_rank', ascending=True).reset_index(drop=True)
    
    pots = [
        df_sorted.iloc[0:6]['team_name'].tolist(),   
        df_sorted.iloc[6:12]['team_name'].tolist(),  
        df_sorted.iloc[12:18]['team_name'].tolist(), 
        df_sorted.iloc[18:24]['team_name'].tolist()  
    ]
    
    for pot in pots:
        random.shuffle(pot)
        
    groups = {
        'A': [], 'B': [], 'C': [], 
        'D': [], 'E': [], 'F': []
    }
    group_keys = list(groups.keys())
    
    for pot in pots:
        for i, team in enumerate(pot):
            groups[group_keys[i]].append(team)
            
    return groups

In [56]:
def run_group_stage_iteration(groups_dict, lookup_table, chaos_std, tracker_df):

    advanced_teams = []
    third_place_pool = []
    
    for group_label, teams in groups_dict.items():
        standings = {
            team: {'Points': 0, 'GD': 0, 'GF': 0} 
            for team in teams
        }
        
        matchups = [
            (teams[0], teams[1]), (teams[2], teams[3]),
            (teams[0], teams[2]), (teams[1], teams[3]),
            (teams[0], teams[3]), (teams[1], teams[2])
        ]
        
        for team_1, team_2 in matchups:
            base_gd = lookup_table[team_1][team_2]
            
            sim_gd = base_gd + stats.norm.rvs(loc=0, scale=chaos_std)
            
            t1_goals = max(0, round((2.5 + sim_gd) / 2))
            t2_goals = max(0, round((2.5 - sim_gd) / 2))
            actual_gd = t1_goals - t2_goals
            
            tracker_df.at[team_1, 'Total_Goals_Scored'] += t1_goals
            tracker_df.at[team_2, 'Total_Goals_Scored'] += t2_goals
            
            standings[team_1]['GF'] += t1_goals
            standings[team_1]['GD'] += actual_gd
            standings[team_2]['GF'] += t2_goals
            standings[team_2]['GD'] -= actual_gd
            
            if t1_goals > t2_goals:
                standings[team_1]['Points'] += 3
            elif t2_goals > t1_goals:
                standings[team_2]['Points'] += 3
            else:
                standings[team_1]['Points'] += 1
                standings[team_2]['Points'] += 1
                
        sorted_group = sorted(
            standings.items(), 
            key=lambda x: (x[1]['Points'], x[1]['GD'], x[1]['GF']), 
            reverse=True
        )
        
        advanced_teams.append(sorted_group[0][0])
        advanced_teams.append(sorted_group[1][0])
        
        third_place_pool.append({
            'team_name': sorted_group[2][0],
            'Points': sorted_group[2][1]['Points'],
            'GD': sorted_group[2][1]['GD'],
            'GF': sorted_group[2][1]['GF']
        })
        
        fourth_place_team = sorted_group[3][0]
        tracker_df.at[fourth_place_team, 'Eliminated_Group_Stage'] += 1

    sorted_wildcards = sorted(
        third_place_pool, 
        key=lambda x: (x['Points'], x['GD'], x['GF']), 
        reverse=True
    )
    
    for entry in sorted_wildcards[:4]:
        advanced_teams.append(entry['team_name'])
        
    for entry in sorted_wildcards[4:]:
        tracker_df.at[entry['team_name'], 'Eliminated_Group_Stage'] += 1
        
    return advanced_teams

In [57]:
def simulate_knockout_round(matchups_list, lookup_table, chaos_std,tracker_df):

    winners = []
    losers = []
    
    for team_1, team_2 in matchups_list:
        base_gd = lookup_table[team_1][team_2]
        sim_gd = base_gd + stats.norm.rvs(loc=0, scale=chaos_std)
        
        t1_goals = max(0, round((2.5 + sim_gd) / 2))
        t2_goals = max(0, round((2.5 - sim_gd) / 2))
        
        tracker_df.at[team_1, 'Total_Goals_Scored'] += t1_goals
        tracker_df.at[team_2, 'Total_Goals_Scored'] += t2_goals
        
        if abs(sim_gd) < 0.2: 
            if np.random.rand() > 0.5:
                winners.append(team_1)
                losers.append(team_2)
            else:
                winners.append(team_2)
                losers.append(team_1)
                
        elif sim_gd > 0:
            winners.append(team_1)
            losers.append(team_2)
        else:
            winners.append(team_2)
            losers.append(team_1)
            
    return winners, losers

In [58]:
def run_knockout_stage(seeded_16_teams, lookup_table, chaos_std, tracker_df):

    r16_matchups = [
        (seeded_16_teams[0], seeded_16_teams[15]), (seeded_16_teams[7], seeded_16_teams[8]),
        (seeded_16_teams[3], seeded_16_teams[12]), (seeded_16_teams[4], seeded_16_teams[11]),
        (seeded_16_teams[1], seeded_16_teams[14]), (seeded_16_teams[6], seeded_16_teams[9]),
        (seeded_16_teams[2], seeded_16_teams[13]), (seeded_16_teams[5], seeded_16_teams[10])
    ]
    qf_teams, r16_losers = simulate_knockout_round(r16_matchups, lookup_table, chaos_std,tracker_df)
    for team in r16_losers: tracker_df.at[team, 'Eliminated_R16'] += 1
            

    qf_matchups = [
        (qf_teams[0], qf_teams[1]), (qf_teams[2], qf_teams[3]),
        (qf_teams[4], qf_teams[5]), (qf_teams[6], qf_teams[7])
    ]
    sf_teams, qf_losers = simulate_knockout_round(qf_matchups, lookup_table, chaos_std,tracker_df)
    for team in qf_losers: tracker_df.at[team, 'Eliminated_QF'] += 1


    sf_matchups = [(sf_teams[0], sf_teams[1]), (sf_teams[2], sf_teams[3])]
    finalists, sf_losers = simulate_knockout_round(sf_matchups, lookup_table, chaos_std,tracker_df)
    

    third_place_match = [(sf_losers[0], sf_losers[1])]
    bronze_winner, bronze_loser = simulate_knockout_round(third_place_match, lookup_table, chaos_std,tracker_df)
    
    tracker_df.at[bronze_winner[0], 'Third_Place'] += 1
    tracker_df.at[bronze_loser[0], 'Fourth_Place'] += 1


    champion_list, runner_up_list = simulate_knockout_round([(finalists[0], finalists[1])], lookup_table, chaos_std,tracker_df)
    
    champion = champion_list[0]
    runner_up = runner_up_list[0]
    
    tracker_df.at[runner_up, 'Runner_Up'] += 1
    tracker_df.at[champion, 'Champion'] += 1
    
    return champion

In [59]:
def run_monte_carlo(master_df, prediction_engine, chaos_std, iterations=10000):
    print(f"Running for {iterations} ")
    start_time = time.time()
    
    lookup_table = build_prediction_matrix(master_df, prediction_engine)
    
    lightweight_teams = master_df[['team_name', 'fifa_rank']].copy()
    
    tracker_df = pd.DataFrame(
        0, 
        index=master_df['team_name'], 
        columns=[
            'Total_Goals_Scored', 'Eliminated_Group_Stage', 'Eliminated_R16',
            'Eliminated_QF', 'Fourth_Place', 'Third_Place', 'Runner_Up', 'Champion'
        ]
    )
    
    for i in range(1, iterations + 1):
        current_groups = draw_groups(lightweight_teams) 
        
        advancing_16 = run_group_stage_iteration(current_groups, lookup_table, chaos_std, tracker_df)
        
        champion = run_knockout_stage(advancing_16, lookup_table, chaos_std, tracker_df)
            
    tracker_df['Avg_Goals_Per_Tournament'] = tracker_df['Total_Goals_Scored'] / iterations
    
    cols_to_percent = [
        'Eliminated_Group_Stage', 'Eliminated_R16', 'Eliminated_QF', 
        'Fourth_Place', 'Third_Place', 'Runner_Up', 'Champion'
    ]
    
    for col in cols_to_percent:
        tracker_df[col] = (tracker_df[col] / iterations) * 100
        

    tracker_df['Avg_Matches_Played'] = (
        (tracker_df['Eliminated_Group_Stage'] / 100) * 3 +
        (tracker_df['Eliminated_R16'] / 100) * 4 +
        (tracker_df['Eliminated_QF'] / 100) * 5 +
        (tracker_df['Fourth_Place'] / 100) * 7 +
        (tracker_df['Third_Place'] / 100) * 7 +
        (tracker_df['Runner_Up'] / 100) * 7 +
        (tracker_df['Champion'] / 100) * 7
    )
    
    tracker_df['Avg_Goals_Per_Match'] = tracker_df['Avg_Goals_Per_Tournament'] / tracker_df['Avg_Matches_Played']
    
    tracker_df = tracker_df.drop(columns=['Total_Goals_Scored', 'Avg_Matches_Played'])
    
    final_results = tracker_df.sort_values(by=['Champion', 'Runner_Up', 'Avg_Goals_Per_Match'], ascending=[False, False, False])
    
    final_results = final_results.round(2)
    
    rename_mapping = {
        'Eliminated_Group_Stage': 'Group Stage',
        'Eliminated_R16': 'R16',
        'Eliminated_QF': 'QF',
        'Fourth_Place': '4th Place',
        'Third_Place': '3rd Place',
        'Runner_Up': 'Runner-Up',
        'Champion': 'Winner',
        'Avg_Goals_Per_Tournament': 'Goals/Tourney',
        'Avg_Goals_Per_Match': 'Goals/Match'
    }
    
    final_results = final_results.rename(columns=rename_mapping)
    
    end_time = time.time()
    print(f"\nTotal execution time: {round(end_time - start_time, 2)} seconds.")
    
    return final_results

### To be honest I am not very pleased with the result

### I suspect Argentina and Portugal are being the favourites due to being heavily boosted by Messi and Ronaldo overperforming in MLS and Saudi

### Turkey being 5th I assume is due to me adding coefficient of league and country strength. I suspect my Model learned to favour Teams with a Balanced Z-score being 0 and my coefficients probably brought them and many teams close to that balance 

### In v2 I would  use Z-Scores for only Clustering. I would also have liked to include Pitch Heatmaps and I plan to use statsbomb for my next Version to try get more data 

### My 900 Minute Filter also seemed to hurt my data due to dropping a lot of players in Each Team

In [60]:
final_tournament_probabilities = run_monte_carlo(
    master_df=df,
    prediction_engine=prediction_engine, 
    chaos_std=chaos_std, 
    iterations=100000
)

display(final_tournament_probabilities)

Running for 100000 

Total execution time: 832.84 seconds.


,Group Stage,R16,QF,4th Place,3rd Place,Runner-Up,Winner,Goals/Tourney,Goals/Match
team_name,,,,,,,,,
Argentina,9.84,33.40,22.58,5.54,8.60,8.54,11.50,8.16,1.58
Portugal,10.93,34.06,22.33,5.64,8.31,8.01,10.72,7.94,1.56
Brazil,11.72,33.97,22.20,5.60,8.10,8.07,10.33,7.83,1.54
England,10.10,34.31,23.28,5.60,8.21,8.23,10.28,8.00,1.57
Türkiye,19.34,31.80,20.10,5.25,7.15,7.31,9.06,7.03,1.44
Germany,17.86,34.17,21.14,5.48,6.71,6.92,7.72,6.94,1.44
France,14.05,36.20,22.44,5.73,7.04,6.93,7.61,7.28,1.48
Spain,14.96,36.21,22.18,5.75,6.82,6.75,7.34,7.16,1.47
Belgium,21.27,35.49,20.44,5.26,5.74,5.94,5.86,6.46,1.38
